In [1]:
import numpy as np
import pandas as pd
import math
import scipy.linalg as linalg

from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import cm
from scipy.integrate import quad,dblquad,odeint

In [ ]:
#测试一重积分
def ts_quad():
    def myfunc(x,n):
        return x ** n
    def advReward(x):
        return 1000000 *(np.exp(0.02 * x))

    # print(quad(myfunc,0,2,args=(2,))) #args中指定了参数n
    revenue = quad(advReward,0,12)[0]
    print(revenue)
#测试二重积分
def ts_dblquad():
    def f(x,y):
        return 3 * (x**2) * (y**2)
    def h(x):
        return 1 - x**2
    v,err = dblquad(f,0,1,0,h)
    print(v)
#洛伦兹方程(蝴蝶效应)：
def ts_lorenz():
    def lorenz(w,t,p,r,b):
        #位置矢量w，三个参数p，r，b
        x,y,z = w.tolist()
        #分别计算dx/dt  dy/dt dz/dt
        return p * (y-x), x * (r-z)-y,x*y - b*z

    fig = plt.figure()
    ax = fig.gca(projection='3d')
    t = np.arange(0,30,0.02)
    initial_val = (0.0,1.00,0.0)
    track = odeint(lorenz,initial_val,t,args=(10.0,28.0,3.0))
    x,y,z = track[:,0],track[:,1],track[:,2]
    ax.plot(x,y,z,label='lorenz')
    ax.legend()
    plt.show()

def ts_odeint():
    def model(y,t,n):
        tau = 10.0 / float(n)
        dydt = np.empty(n)

        dydt[0] = (1.0/tau)*(-y[0]+1.0)
        for i in range(1,n):
            dydt[i] = (1.0 / tau) * (-y[i] + y[i-1])

        return dydt
    t = np.linspace(0,20,100)
    n = 100
    y0 = np.zeros(n)

    y = odeint(model,y0,t,args=(n,))
    plt.plot(t,y,'r--',linewidth=2)
    plt.plot([0,10,10.0001,20],[0,0,1,1],'k-',linewidth=3)
    plt.show()

In [ ]:
# 3维向量的旋转：
def invoke_rotate_3d(angle,A,spin_axis=(0,0,1)):
    # 3维向量的旋转：
    def rotate_3d(axis, radian):
        """
        :param axis: 旋转轴
        :param radian: 旋转的弧度
        :return:
        """
        rot_matrix = linalg.expm(np.cross(np.eye(3), axis / linalg.norm(axis) * radian))
        return rot_matrix

    # 分别是x,y和z轴,也可以自定义旋转轴
    # axis_x, axis_y, axis_z = [1, 0, 0], [0, 1, 0], [0, 0, 1]

    # rand_axis = [1, 1, 0]
    # 旋转角度
    # yaw = math.pi / 180
    rot_matrix = rotate_3d(spin_axis, angle/180 * np.pi)
    print(rot_matrix)
    print(rot_matrix@A)
#使用expm来进行在指定的平面中旋转
def invoke_rotate_expm():
    def rotate_expm(angle, A, B):
        B2 = v_shadow(A,B)   #B * np.linalg.norm(A) / np.linalg.norm(B)
        B2A = B2 - A
        base2 = np.array([[0, B2A[1], B2A[2]], [B2A[0], 0, B2A[2]], [B2A[0], B2A[1], 0]])
        base2 = base2.T
        print(base2)
        return linalg.expm(base2 * angle) @ A

    A = np.array([1,1,0])
    B = np.array([0,0,1])
    angle = v_angle(A,B)[0]
    print(rotate_expm(np.pi/4,A,B))




#围绕任意轴，对向量进行一个旋转指定角度
def rotate(angle,A,*spin_axis):
    """
    :param angle:旋转的角度，非弧度
    :param A: 需要旋转的向量
    :param spin_axis: 指定的转动轴
    :return:
    """
    if spin_axis is not None:
        # spin_axis = spin_axis[0]    #?
        spin_axis = spin_axis / np.linalg.norm(spin_axis)   #
    else:
        #如果没有指定转动轴，那么就是使用最后一个维度作为转动轴
        spin_axis = np.zeros(len(A))
        spin_axis[-1] = 1
    #先得到旋转平面和旋转轴的 基坐标：axis_x，axis_y，axis_z，其中axis_z 和旋转轴对应
    axis_z = spin_axis / np.linalg.norm(spin_axis)
    B = axis_z * (A@spin_axis / np.linalg.norm(spin_axis))  #转动轴
    C = A - B  #垂直于转动轴B 的向量
    axis_y = C / np.linalg.norm(C)
    axis_x = np.cross(axis_z,axis_y)
    #旋转基对应新坐标基中的表示：
    angle = angle / 180 * np.pi
    new_base = np.column_stack((axis_x.reshape(-1, 1), axis_y.reshape(-1, 1), axis_z.reshape(-1, 1)))  #新的基
    # new_base_I = np.linalg.inv(new_base)
    C2 = np.linalg.inv(new_base)@C

    rotate_base = np.array([[np.cos(angle),-np.sin(angle),0],[np.sin(angle),np.cos(angle),0],[0,0,1]])
    C3 = new_base@(rotate_base@C2)  #在新基中旋转后的C
    A3 = B + C3   #在新基中旋转后的向量A

    return A,B,C,C3,A3,new_base,rotate_base

In [ ]:
# 向量A向B投影 ，返回的是与B向量平行的模长是|A|cosθ
def v_shadow(A, B):
    return B * (A @ B / (np.linalg.norm(B) ** 2))
#任意多的维度中，一个向量投影到一个平面：
def p2a(P, A, B, C=None):  # 点P到平面的距离和投影的点
    if C is not None:   #使用3个点确定一个面，如果为空表示使用两个向量确定一个面
        B = C - A
        A = B - A
    P1 = v_shadow(A, P)  # 新的Y向量：A向量在y向量上的投影向量
    P2 = v_shadow(B, P)  # B向量在y向量上的投影向量
    # PA_new = P1 - A  # A向量 到y1 向量 的向量
    P1B_new = ((P2 - B) * np.linalg.norm(P1)) / (np.linalg.norm(P2))  # P1B_new 是在P1点的垂线，而且平行于P2B
    # 如果P1，P2同方向，求B_new就是加，如果不同方向，就是减
    if v_angle(P1,P2)[1] < 1 : #表示P1，P2同方向
        B_new = P1 - P1B_new
    else:
        B_new = P1 + P1B_new  # 根据P1B_new 找到B_new，不知道P1，P2两点在同一边时是否是用这个
    AB_new = A - B_new  # 从B_new到A 的向量,AB_new也垂直于P


    # Y_hat = B_new + v_shadow(YB_new,AB_new)
    # t = v_shadow(B_new, AB_new)  # 和AB_new是反向的,它也和y垂直
    # print(v_angle(t,B_new),np.linalg.norm(t),np.linalg.norm(B_new))
    Y_hat = B_new - v_shadow(B_new, AB_new)
    # print(v_angle(t,B_new),v_angle(Y_hat,t),v_angle(Y_hat,B_new))
    # print(np.linalg.norm(Y_hat)**2,np.linalg.norm(B_new)**2 - np.linalg.norm(t)**2)
    # print(Y_hat@t)
    # Y_hat_new = v_shadow(y1, Y_hat)
    P_hat = v_shadow(P, Y_hat)
    return P_hat,P - P_hat